In [1]:
import pandas as pd
import math, html, re

In [2]:
# --- Load and keep only relevant columns ---
df = pd.read_csv('../input_data/carolina_schedule.csv')
columns_to_keep = ['Day', 'Time (EDT)', 'Location', 'Event', 'Speaker']
df = df[columns_to_keep].copy()

# Normalize Day values (strip/keep original for display)
df['Day'] = df['Day'].astype(str).str.strip()

def safe(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return '--'
    return html.escape(str(v), quote=True)

# Extract weekday key from "Monday" or "Monday, Sept. 15", etc.
def day_key(s):
    m = re.search(r'(monday|tuesday|wednesday|thursday|friday)', str(s), re.I)
    return m.group(1).lower() if m else str(s).strip().lower()

# Order + CSS class + anchor id per weekday
order_keys   = ['monday','tuesday','wednesday','thursday','friday']
display_name = {k: k.capitalize() for k in order_keys}
day_class    = {
    'monday':    'day-observations',  # green
    'tuesday':   'day-theory',        # duke blue
    'wednesday': 'day-sims',          # orange
    'thursday':  'day-hack',          # purple
    'friday':    'day-hack',          # purple
}

# Find which days exist in the CSV (in our preferred order)
present = []
keys_in_csv = df['Day'].apply(day_key)
for k in order_keys:
    if (keys_in_csv == k).any():
        present.append(k)

table_headers = ['Time (EDT)', 'Location', 'Event', 'Speaker']
tables_html = []

for k in present:
    day_df = df[keys_in_csv == k]
    if day_df.empty:
        continue

    cls   = day_class.get(k, 'day-default')
    aid   = k  # anchor like id="monday"
    title = display_name.get(k, k.capitalize())

    # build rows
    rows = []
    for _, r in day_df.iterrows():
        cells = ''.join(f"<td>{safe(r[col])}</td>" for col in table_headers)
        rows.append(f"<tr>{cells}</tr>")

    table_html = f"""
<h2 id="{aid}" class="schedule-day {cls}">{html.escape(title)}</h2>
<table class="schedule-table {cls}">
  <thead>
    <tr>
      {''.join(f'<th>{html.escape(h)}</th>' for h in table_headers)}
    </tr>
  </thead>
  <tbody>
    {''.join(rows)}
  </tbody>
</table>
""".strip()
    tables_html.append(table_html)

html_schedule = "\n\n".join(tables_html)
print(html_schedule)


<h2 id="monday" class="schedule-day day-observations">Monday</h2>
<table class="schedule-table day-observations">
  <thead>
    <tr>
      <th>Time (EDT)</th><th>Location</th><th>Event</th><th>Speaker</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>09:00–09:30</td><td>Room 1 or Lounge</td><td>Coffee &amp; Arrival</td><td>--</td></tr><tr><td>09:30–10:00</td><td>Room 1</td><td>Welcome &amp; Overview</td><td>Sarcevic &amp; Troxel</td></tr><tr><td>10:00–11:00</td><td>Room 1</td><td>Brief Introductions</td><td>all</td></tr><tr><td>11:00–11:30</td><td>Room 1</td><td>Observations Overview</td><td>TBA</td></tr><tr><td>11:30–12:30</td><td>Room 1</td><td>Flash Talks: Observational Inputs</td><td>--</td></tr><tr><td>12:30–13:30</td><td>Hallway</td><td>Lunch</td><td>--</td></tr><tr><td>13:30–14:30</td><td>Room 1</td><td>Project Pitch Block I</td><td>TBA</td></tr><tr><td>14:30–17:00</td><td>Room 2 / Lounge / Terrace</td><td>Discussion + Team Formation</td><td>all</td></tr>
  </tbody>
</table>

<h2 